# 04 SurgeDesk coordinator demo
Recommendations only. A human makes every dispatch decision. Saved local outputs are development evidence; LinuxONE confirmation is pending.

In [1]:
# Bootstrap: make the repo importable whether the notebook runs in IBM's container or ours
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from surgedesk.envcheck import run
ENV = run()

SurgeDesk environment check
arch            : arm64  (byte order: little)
python          : 3.13.0
cores / RAM     : 8 / None GB
disk free       : 29.9 GB
NNPA flag       : None
zDNN libs       : none found
ROOT            : /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk
threads setting : 2   demo event: hurricane_harvey_2017
----------------------------------------------------------------
  numpy              2.1.2
  pandas             2.2.3
  scikit-learn       1.7.0
  xgboost            NOT INSTALLED
  torch              2.9.0
  transformers       4.46.2
  tokenizers         0.20.3
  huggingface_hub    0.33.1
  llama_cpp_python   NOT INSTALLED
  ipywidgets         8.1.7
  matplotlib         3.10.3
  joblib             1.5.1
  torch_nnpa         NOT INSTALLED
  nbconvert          7.17.2
----------------------------------------------------------------
  --  humaid_dir     /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk/data/raw/humaid
  --  gazetteer_dir  /U

In [2]:
STUB = False  # Set True for clearly labelled synthetic fixture cards.
from surgedesk import config, fileio, ui
from surgedesk.pipeline import StageUnavailable
from IPython.display import display, HTML
if STUB:
    from surgedesk.contracts import Report
    from surgedesk.stubs import run_stub_pipeline
    reports = fileio.read_jsonl(config.FIXTURES / "outputs" / config.FIXTURE_EVENT / config.REPORTS_FILE, Report)
    run_stub_pipeline(config.FIXTURE_EVENT, reports)
    print("Synthetic fixture demonstration; scores are stubs, not measured model results.")

The coordinator reviews ranked incident cards, quoted Granite evidence, rejected sentences, and masked member reports. Moving λ reorders stored scores without model calls. Accept, Override, and Dismiss append validated decisions. Replay shows historical tweets with compressed time; it does not represent current emergencies.

In [3]:
if not STUB:
    try:
        replay_view = ui.live_replay(speed=5.0, duration_s=60)
        display(replay_view)
    except (StageUnavailable, FileNotFoundError) as exc:
        print(f"Replay not available yet: {exc}")
else:
    print("Historical replay skipped in synthetic fixture mode.")

Replay not available yet: stage live replay not implemented yet (owner: Agent 1): module replay or its dependency is unavailable (No module named 'surgedesk.replay')


In [4]:
try:
    display(ui.dashboard(stub=STUB))
except (ImportError, StageUnavailable, FileNotFoundError) as exc:
    print(f"Dashboard not available yet: {exc}")
    try:
        display(HTML(ui.render_html(stub=STUB)))
    except (StageUnavailable, FileNotFoundError) as fallback_error:
        print(f"Cards not available yet: {fallback_error}")

Dashboard not available yet: /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk/outputs/hurricane_harvey_2017/incidents.jsonl does not exist. Run the stage that produces it first.
Cards not available yet: /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk/outputs/hurricane_harvey_2017/incidents.jsonl does not exist. Run the stage that produces it first.


Static fallback with the same evidence cards, suitable when widget support is unavailable:

In [5]:
try:
    display(HTML(ui.render_html(stub=STUB)))
except (StageUnavailable, FileNotFoundError) as exc:
    print(f"Static cards not available yet: {exc}")

Static cards not available yet: /Users/saikarthikbattula/Documents/ChatGPT/Person1/surgedesk/outputs/hurricane_harvey_2017/incidents.jsonl does not exist. Run the stage that produces it first.


In [6]:
from surgedesk.contracts import Decision
from surgedesk.textutil import mask_privacy
event = config.FIXTURE_EVENT if STUB else config.DEMO_EVENT
decisions_path = config.event_file(event, config.DECISIONS_FILE, stub=STUB)
if decisions_path.exists():
    for decision in fileio.read_jsonl(decisions_path, Decision)[-5:]:
        print(mask_privacy(__import__("json").dumps(decision.to_dict())))
else:
    print("No review decisions recorded yet.")

No review decisions recorded yet.
